# Notebook 2 — Statistical Comparison of Anomaly Detection Models

This notebook performs the inferential statistical comparison between LGMMA-X and the six baseline models using the frozen chronological test set.

The analysis uses common chronological evaluation blocks as the paired statistical units. The same test sequences are assigned to the same evaluation block for every model, ensuring that model comparisons are paired on identical portions of the test set.

Positive performance differences indicate that LGMMA-X achieved a higher metric value than the corresponding baseline.

No model fitting, hyperparameter tuning, threshold selection, or test-set optimization is performed in this notebook.

## 1. Statistical Comparison Design

The frozen test set contains 21,858 sequences. To obtain repeated paired observations without creating artificial repetitions, the test sequences are divided chronologically into 10 approximately equal-sized evaluation blocks.

For each baseline and target metric:

$$
d_i = M_{\text{LGMMA-X},i} - M_{\text{Baseline},i}
$$
where \(i\) denotes the common chronological evaluation block.

The paired differences are tested for normality using the Shapiro-Wilk test.

- If the paired differences are approximately normal (`p >= 0.05`), a paired t-test is used.
- Otherwise, the Wilcoxon signed-rank test is used.

Because six baselines are compared for each metric, Holm-Bonferroni correction is applied separately within each metric at \(\alpha = 0.05\).

The raw reconstruction-error ranking ablation is threshold-free and therefore contributes only ROC-AUC and PR-AUC.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import shapiro, ttest_rel, wilcoxon
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

## 2. Paths and Frozen Evaluation Artifacts

All model outputs used here are frozen artifacts generated before statistical comparison.

The notebook does not retrain models or select thresholds.

In [ ]:
REPO_DIR = Path.cwd()

if REPO_DIR.name == "notebooks":
    SRC_DIR = REPO_DIR.parent
else:
    SRC_DIR = REPO_DIR / "src"

MODEL_DATA_DIR = SRC_DIR / "modeling" / "data"
COMPARISON_DIR = SRC_DIR / "baseline" / "statistical_comparison"

TEST_METADATA_PATH = (
    MODEL_DATA_DIR
    / "sequence_metadata"
    / "test_sequence_metadata.parquet"
)

TEST_LABELS_PATH = (
    MODEL_DATA_DIR
    / "evaluation"
    / "test_proxy_labels.csv"
)

GMM_SCORES_PATH = (
    MODEL_DATA_DIR
    / "gmm_results"
    / "test_anomaly_scores.csv"
)

GMM_THRESHOLD_PATH = (
    MODEL_DATA_DIR
    / "evaluation"
    / "selected_threshold.txt"
)

MODEL_FILES = {
    "LGMMA-X": GMM_SCORES_PATH,
    "ARIMA-GARCH": COMPARISON_DIR / "arima_garch_test_scores.csv",
    "Z-score": COMPARISON_DIR / "zscore_test_scores.csv",
    "Isolation Forest": COMPARISON_DIR / "isolation_forest_test_scores.csv",
    "One-Class SVM": COMPARISON_DIR / "one_class_svm_test_scores.csv",
    "LSTM-AE (Fixed Threshold)": COMPARISON_DIR / "lstm_fixed_test_scores.csv",
    "LSTM-AE (Raw Error Ranking)": COMPARISON_DIR / "lstm_rank_test_scores.csv",
}

BASELINES = list(MODEL_FILES)[1:]

BINARY_METRICS = [
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
]

SCORE_METRICS = [
    "roc_auc",
    "pr_auc",
]

TARGET_METRICS = BINARY_METRICS + SCORE_METRICS

N_BLOCKS = 10

STANDARD_COLUMNS = [
    "sequence_id",
    "segment_id",
    "start_timestamp",
    "end_timestamp",
    "anomaly_score",
    "proxy_anomaly",
]

BINARY_COLUMNS = STANDARD_COLUMNS + ["prediction"]

## 3. Load LGMMA-X Test Scores

LGMMA-X scores are reconstructed from the frozen GMM anomaly scores, test sequence metadata, proxy labels, and the frozen validation-calibrated threshold.

The threshold is not recalculated on the test set.

In [ ]:
def load_gmm_scores():
    metadata = pd.read_parquet(TEST_METADATA_PATH)[
        [
            "sequence_id",
            "segment_id",
            "start_timestamp",
            "end_timestamp",
        ]
    ]

    labels = pd.read_csv(TEST_LABELS_PATH)[
        ["sequence_id", "proxy_anomaly"]
    ]

    scores = pd.read_csv(GMM_SCORES_PATH)[
        ["sequence_id", "anomaly_score"]
    ]

    output = metadata.merge(
        scores,
        on="sequence_id",
        validate="one_to_one",
    )

    output = output.merge(
        labels,
        on="sequence_id",
        validate="one_to_one",
    )

    threshold = float(
        GMM_THRESHOLD_PATH.read_text(
            encoding="utf-8"
        ).strip()
    )

    output["prediction"] = (
        output["anomaly_score"] >= threshold
    ).astype(int)

    return output


lgmma_x = load_gmm_scores()

print(f"LGMMA-X test sequences: {len(lgmma_x):,}")
print(
    f"LGMMA-X threshold: "
    f"{float(GMM_THRESHOLD_PATH.read_text(encoding='utf-8').strip()):.6f}"
)

## 4. Load Standardized Baseline Score Files

Binary baselines must contain a frozen prediction generated using their previously selected validation threshold.

The LSTM-AE Raw Error Ranking ablation is intentionally treated differently because it is threshold-free and contains only continuous anomaly scores.

In [ ]:
def load_baseline_scores(model_name, path):
    output = pd.read_csv(path)

    required = set(STANDARD_COLUMNS)

    if model_name != "LSTM-AE (Raw Error Ranking)":
        required.add("prediction")

    missing = required - set(output.columns)

    if missing:
        raise ValueError(
            f"{model_name} is missing required columns: "
            f"{sorted(missing)}"
        )

    columns = (
        BINARY_COLUMNS
        if model_name != "LSTM-AE (Raw Error Ranking)"
        else STANDARD_COLUMNS
    )

    return output[columns].copy()


missing_files = [
    path
    for path in MODEL_FILES.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following frozen score files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )


score_tables = {
    "LGMMA-X": lgmma_x
}

for model, path in MODEL_FILES.items():
    if model == "LGMMA-X":
        continue

    score_tables[model] = load_baseline_scores(
        model,
        path,
    )

print(f"Loaded {len(score_tables)} models.")

## 5. Validate Cross-Model Alignment

Every model must contain exactly the same test sequence IDs.

This ensures that comparisons are performed on identical chronological observations.

The proxy labels are also required to be binary and consistent across models.

In [ ]:
reference = score_tables["LGMMA-X"].copy()

reference_ids = set(reference["sequence_id"])

for model, table in score_tables.items():

    if table["sequence_id"].duplicated().any():
        raise ValueError(
            f"{model} contains duplicate sequence_id values."
        )

    if set(table["sequence_id"]) != reference_ids:
        raise ValueError(
            f"{model} does not share the exact "
            "LGMMA-X sequence IDs."
        )

    if not table["proxy_anomaly"].isin([0, 1]).all():
        raise ValueError(
            f"{model} contains invalid proxy labels."
        )

print("Cross-model sequence alignment validated.")
print(f"Common test sequences: {len(reference_ids):,}")

## 6. Create Common Chronological Evaluation Blocks

The test set is sorted chronologically using the sequence start timestamp and sequence ID.

Ten approximately equal-sized chronological blocks are then created.

The resulting block assignment is based only on the frozen test sequence ordering and is shared by every model.

`segment_id` is not used as the statistical unit because it represents continuity between timestamps rather than a standardized evaluation replicate.

In [ ]:
reference = (
    reference
    .sort_values(
        ["start_timestamp", "sequence_id"]
    )
    .reset_index(drop=True)
)

reference["evaluation_block"] = (
    np.arange(len(reference)) * N_BLOCKS // len(reference)
) + 1

block_mapping = reference[
    ["sequence_id", "evaluation_block"]
].copy()

for model, table in score_tables.items():

    score_tables[model] = (
        table
        .merge(
            block_mapping,
            on="sequence_id",
            validate="one_to_one",
        )
        .sort_values(
            ["evaluation_block", "start_timestamp", "sequence_id"]
        )
        .reset_index(drop=True)
    )

block_counts = (
    reference["evaluation_block"]
    .value_counts()
    .sort_index()
)

display(block_counts.to_frame("sequence_count"))

print(
    f"Evaluation blocks: {N_BLOCKS}"
)
print(
    f"Total sequences: {len(reference):,}"
)

## 7. Calculate Block-Level Performance Metrics

Binary metrics are calculated only for models with a frozen binary prediction.

For the threshold-free LSTM-AE Raw Error Ranking ablation, only ROC-AUC and PR-AUC are calculated.

In [ ]:
def calculate_block_metrics(model_name, table):
    rows = []

    for block_id, block in table.groupby(
        "evaluation_block",
        sort=True,
    ):
        labels = block["proxy_anomaly"].to_numpy(dtype=int)
        scores = block["anomaly_score"].to_numpy(dtype=float)

        row = {
            "evaluation_block": block_id,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "balanced_accuracy": np.nan,
            "roc_auc": np.nan,
            "pr_auc": np.nan,
        }

        # Binary metrics are not applicable to
        # the threshold-free raw ranking ablation.
        if model_name != "LSTM-AE (Raw Error Ranking)":
            predictions = block["prediction"].to_numpy(dtype=int)

            row["precision"] = precision_score(
                labels,
                predictions,
                zero_division=0,
            )

            row["recall"] = recall_score(
                labels,
                predictions,
                zero_division=0,
            )

            row["f1"] = f1_score(
                labels,
                predictions,
                zero_division=0,
            )

            row["balanced_accuracy"] = (
                balanced_accuracy_score(
                    labels,
                    predictions,
                )
            )

        # ROC-AUC and PR-AUC require both classes
        # to be present in the block.
        if np.unique(labels).size == 2:
            row["roc_auc"] = roc_auc_score(
                labels,
                scores,
            )

            row["pr_auc"] = average_precision_score(
                labels,
                scores,
            )

        rows.append(row)

    return pd.DataFrame(rows)


block_metrics = {
    model: calculate_block_metrics(
        model,
        table,
    )
    for model, table in score_tables.items()
}

for model, metrics in block_metrics.items():
    print(
        f"{model}: "
        f"{len(metrics)} evaluation blocks"
    )

## 8. Inspect Block-Level Metrics

The following table provides the block-level observations used for paired statistical testing.

These are not additional model runs. They are measurements obtained by evaluating the same frozen models on common chronological portions of the test set.

In [ ]:
block_metric_table = []

for model, metrics in block_metrics.items():

    temp = metrics.copy()
    temp.insert(0, "model", model)

    block_metric_table.append(temp)

block_metric_table = pd.concat(
    block_metric_table,
    ignore_index=True,
)

display(
    block_metric_table
)

## 9. Construct Paired Performance Differences

For every baseline and target metric:

$d_i = M_{\text{LGMMA-X},i} - M_{\text{Baseline},i}$

Positive differences indicate higher performance by LGMMA-X.

The proposed model and baseline are evaluated on the same chronological block, producing paired observations.

In [ ]:
paired_rows = []

for baseline in BASELINES:

    merged = block_metrics["LGMMA-X"].merge(
        block_metrics[baseline],
        on="evaluation_block",
        suffixes=(
            "_proposed",
            "_baseline",
        ),
        validate="one_to_one",
    )

    for metric in TARGET_METRICS:

        valid = merged[
            [
                "evaluation_block",
                f"{metric}_proposed",
                f"{metric}_baseline",
            ]
        ].dropna()

        for row in valid.itertuples(
            index=False
        ):

            proposed_value = getattr(
                row,
                f"{metric}_proposed",
            )

            baseline_value = getattr(
                row,
                f"{metric}_baseline",
            )

            paired_rows.append(
                {
                    "baseline": baseline,
                    "metric": metric,
                    "evaluation_block": row.evaluation_block,
                    "proposed_value": proposed_value,
                    "baseline_value": baseline_value,
                    "difference": (
                        proposed_value
                        - baseline_value
                    ),
                }
            )

paired_differences = pd.DataFrame(
    paired_rows
)

display(
    paired_differences.head()
)

print(
    f"Paired observations: "
    f"{len(paired_differences):,}"
)

## 10. Statistical Test Selection

The paired differences are first evaluated using the Shapiro-Wilk normality test.

- `p >= 0.05`: paired t-test
- `p < 0.05`: Wilcoxon signed-rank test

If fewer than three valid paired observations are available, inferential testing is considered insufficient.

An all-zero difference vector is handled explicitly because the Wilcoxon signed-rank test is undefined when all differences are zero.

In [ ]:
def compare_pair(differences, alpha=0.05):
    differences = np.asarray(
        differences,
        dtype=float,
    )

    differences = differences[
        np.isfinite(differences)
    ]

    result = {
        "n": len(differences),
        "mean_difference": (
            np.mean(differences)
            if len(differences)
            else np.nan
        ),
        "median_difference": (
            np.median(differences)
            if len(differences)
            else np.nan
        ),
        "shapiro_stat": np.nan,
        "shapiro_p": np.nan,
        "test": "insufficient_data",
        "statistic": np.nan,
        "p_value": np.nan,
    }

    if len(differences) < 3:
        return result

    shapiro_stat, shapiro_p = shapiro(
        differences
    )

    result["shapiro_stat"] = shapiro_stat
    result["shapiro_p"] = shapiro_p

    if shapiro_p >= alpha:

        statistic, p_value = ttest_rel(
            differences,
            np.zeros_like(differences),
        )

        result["statistic"] = statistic
        result["p_value"] = p_value
        result["test"] = "paired_t_test"

    elif np.all(differences == 0):

        result["statistic"] = 0.0
        result["p_value"] = 1.0
        result["test"] = (
            "wilcoxon_signed_rank_all_zero"
        )

    else:

        statistic, p_value = wilcoxon(
            differences,
            alternative="two-sided",
        )

        result["statistic"] = statistic
        result["p_value"] = p_value
        result["test"] = (
            "wilcoxon_signed_rank"
        )

    return result

## 11. Run Paired Statistical Tests

In [ ]:
results = []

for (
    baseline,
    metric,
), group in paired_differences.groupby(
    ["baseline", "metric"]
):

    result = compare_pair(
        group["difference"]
    )

    result.update(
        {
            "baseline": baseline,
            "metric": metric,
        }
    )

    results.append(result)

statistical_results = pd.DataFrame(
    results
)

display(
    statistical_results
)

## 12. Holm-Bonferroni Multiple-Comparison Correction

Six baselines are compared against LGMMA-X for each target metric.

Therefore, Holm-Bonferroni correction is applied separately within each metric across the six baseline comparisons.

The corrected significance level is:

\[
\alpha = 0.05
\]

A comparison is considered statistically significant after correction when its Holm-adjusted p-value is less than or equal to 0.05.

In [ ]:
def holm_bonferroni(
    p_values,
    alpha=0.05,
):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    adjusted = np.full(
        len(p_values),
        np.nan,
    )

    valid_indices = np.flatnonzero(
        np.isfinite(p_values)
    )

    if len(valid_indices) == 0:
        return (
            adjusted,
            np.zeros(
                len(p_values),
                dtype=bool,
            ),
        )

    ordered = valid_indices[
        np.argsort(
            p_values[valid_indices]
        )
    ]

    running_max = 0.0

    for rank, index in enumerate(
        ordered
    ):
        corrected = (
            len(ordered) - rank
        ) * p_values[index]

        running_max = max(
            running_max,
            corrected,
        )

        adjusted[index] = min(
            running_max,
            1.0,
        )

    rejected = (
        np.isfinite(adjusted)
        & (adjusted <= alpha)
    )

    return adjusted, rejected

In [ ]:
statistical_results[
    "holm_p_value"
] = np.nan

statistical_results[
    "significant_after_holm"
] = False

for metric, indices in (
    statistical_results
    .groupby("metric")
    .groups.items()
):

    adjusted, rejected = (
        holm_bonferroni(
            statistical_results.loc[
                indices,
                "p_value",
            ].to_numpy()
        )
    )

    statistical_results.loc[
        indices,
        "holm_p_value",
    ] = adjusted

    statistical_results.loc[
        indices,
        "significant_after_holm",
    ] = rejected

## 13. Statistical Comparison Summary

In [ ]:
summary = statistical_results[
    [
        "metric",
        "baseline",
        "n",
        "mean_difference",
        "median_difference",
        "shapiro_p",
        "test",
        "p_value",
        "holm_p_value",
        "significant_after_holm",
    ]
].sort_values(
    [
        "metric",
        "holm_p_value",
    ]
)

display(summary)

## 14. Distribution of Paired Performance Differences

The following visualization shows the distribution of LGMMA-X minus baseline performance differences across the chronological evaluation blocks.

Values above zero indicate that LGMMA-X performed better for the corresponding block and metric.

In [ ]:
plt.figure(figsize=(14, 7))

sns.boxplot(
    data=paired_differences,
    x="metric",
    y="difference",
    hue="baseline",
    showfliers=False,
)

plt.axhline(
    0,
    color="black",
    linewidth=1,
)

plt.xticks(
    rotation=30,
    ha="right",
)

plt.ylabel(
    "LGMMA-X - baseline"
)

plt.xlabel("Metric")

plt.title(
    "Paired chronological block-level performance differences"
)

plt.legend(
    title="Baseline",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()
plt.show()

## 15. Save Statistical Comparison Outputs

Two frozen analysis artifacts are saved:

1. `paired_block_differences.csv` — block-level paired performance differences.
2. `paired_statistical_comparison.csv` — inferential test results and Holm-Bonferroni-adjusted p-values.

In [ ]:
OUTPUT_DIR = (
    SRC_DIR
    / "baseline"
    / "statistical_comparison"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

paired_differences.to_csv(
    OUTPUT_DIR
    / "paired_block_differences.csv",
    index=False,
)

statistical_results.to_csv(
    OUTPUT_DIR
    / "paired_statistical_comparison.csv",
    index=False,
)

print(
    "Saved:"
)

print(
    OUTPUT_DIR
    / "paired_block_differences.csv"
)

print(
    OUTPUT_DIR
    / "paired_statistical_comparison.csv"
)